### General Setup

In [0]:
import json
from mda_reporting.aggregations.stats_aggregator import StatsAggregator
from mda_reporting.aggregations.histogram import (
    HistogramDuration,
    HistogramCustomWeights,
    HistogramDistance,
)
from mda_reporting.aggregations.histogram2d import (
    Histogram2DDuration,
    Histogram2DCustomWeights,
    Histogram2DDistance,
)
from mda_reporting.core.page import Page
from mda_reporting.core.report import Report
from mda_reporting.events.basic_event import BasicEvent

### Initialization

In [0]:
dbutils.widgets.text("catalog_in", "development")
dbutils.widgets.text("schema_in", "bronze_e2e")
dbutils.widgets.text("catalog_out", "development")
dbutils.widgets.text("schema_out", "gold_e2e")
dbutils.widgets.text("prefix_out", "e2e")
dbutils.widgets.text("meta_schema", "e2e_data_model")
dbutils.widgets.text("meta_catalog", "e2e")
dbutils.widgets.text("reset", "false")

catalog_in = dbutils.widgets.get("catalog_in")
schema_in = dbutils.widgets.get("schema_in")
catalog_out = dbutils.widgets.get("catalog_out")
schema_out = dbutils.widgets.get("schema_out")
prefix_out = dbutils.widgets.get("prefix_out")
meta_schema = dbutils.widgets.get("meta_schema")
meta_catalog = dbutils.widgets.get("meta_catalog")
reset = dbutils.widgets.get("reset")
# gold_target:

if reset == "true":
    spark.sql(
        f"DROP TABLE IF EXISTS {catalog_out}.{schema_out}.{prefix_out}_histogram_fact"
    )
    spark.sql(
        f"DROP TABLE IF EXISTS {catalog_out}.{schema_out}.{prefix_out}_histogram2d_fact"
    )
    spark.sql(
        f"DROP TABLE IF EXISTS {catalog_out}.{schema_out}.{prefix_out}_histogram_dimension"
    )
    spark.sql(
        f"DROP TABLE IF EXISTS {catalog_out}.{schema_out}.{prefix_out}_histogram2d_dimension"
    )
    spark.sql(
        f"DROP TABLE IF EXISTS {catalog_out}.{schema_out}.{prefix_out}_event_instance_fact"
    )
    spark.sql(
        f"DROP TABLE IF EXISTS {catalog_out}.{schema_out}.{prefix_out}_event_dimension"
    )
    spark.sql(
        f"DROP TABLE IF EXISTS {catalog_out}.{schema_out}.{prefix_out}_measurement_dimension"
    )

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_out}.{schema_out}")

In [0]:
config = json.load(open("./config/config_e2e.json"))

config["source"][
    "container_tags_table"
] = f"{meta_catalog}.{meta_schema}.concept_entities"
config["source"][
    "container_metrics_table"
] = f"{catalog_in}.{schema_in}.container_metric"
config["source"]["channel_metrics_table"] = f"{catalog_in}.{schema_in}.channel_metric"
config["source"]["channels_uri"] = f"{catalog_in}.{schema_in}.channel_data"

config["unity_sink"]["catalog"] = catalog_out
config["unity_sink"]["schema"] = schema_out
config["unity_sink"]["table_prefix"] = prefix_out

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "auto")
my_report = Report(name="my_report", spark=spark, config=config)
db = my_report.get_db()

### Signal Queries & Virtual Signal Creation

In [0]:
# physical signals
eng_rpm = db.query.channel(channel_name="is1_eng_speed", data_key="TM")
veh_spd = db.query.channel(channel_name="can_vehicle_speed", data_key="TM")
odo_km = db.query.channel(
    channel_name="can_in_CGW_C1_TotalVehDist_Cval_DIAG", data_key="TM"
)

# event expressions
veh_spd_event = veh_spd > 10

### Event Definitions
All Events are defined in a way where all datapoints return _true_, effectivly leading to a '_Measurement_' event.

In [0]:
veh_spd_event = BasicEvent(
    name="speed_event",
    expr=veh_spd_event,
    desc="Vehicle speed > 10",
    required_channels=["can_vehicle_speed"],
)

my_report.add_event(veh_spd_event)

### Stats Definitions

In [ ]:
# Definition of 1st page
my_first_page = Page(page_number=1)
my_report.add_page(my_first_page)


stats_aggregator = StatsAggregator(
    name="stats_agg",
    input_expressions=[veh_spd, eng_rpm],
    channel_names=["can_vehicle_speed", "is1_eng_speed"],
    statistics=["start", "end", "min", "max", "mean"],
    event=veh_spd_event,
)

my_first_page.add_aggregation(stats_aggregator)

### Histogram Definition

The ranges for the histograms where set to reproduce the same bin values as the legacy solution which uses automatic bin definition via bin size alone.

In [0]:
# Speed histogram within rpm events
hist1_name = "speed_hist"
hist1_desc = "Vehicle speed histogram within speed events"
hist1_bins = [float(i) for i in range(0, 100, 5)]
hist1 = HistogramDuration(
    name=hist1_name,
    base_expr=veh_spd,
    event=veh_spd_event,
    bins=hist1_bins,
    desc=hist1_desc,
    channel_name="can_vehicle_speed",
)
my_first_page.add_aggregation(hist1)

hist2_name = "rpm_hist"
hist2_desc = "Engine speed histogram within speed events"
hist2_bins = [float(i) for i in range(0, 2400, 50)]
hist2 = HistogramDuration(
    name=hist2_name,
    base_expr=eng_rpm,
    event=veh_spd_event,
    bins=hist2_bins,
    desc=hist2_desc,
    channel_name="is1_eng_speed",
)
my_first_page.add_aggregation(hist2)


hist_6_name = "histogram_1d_distance"
hist_6_desc = "histogram 1d distance values"
hist_6_bins = [float(i) for i in range(0, 120, 5)]
hist_6 = HistogramDistance(
    name=hist_6_name,
    base_expr=veh_spd,
    event=veh_spd_event,  # speed_above_0_event,
    weights_expr=odo_km,
    bins=hist_6_bins,
    desc=hist_6_desc,
    channel_name="can_in_CGW_C1_TotalVehDist_Cval_DIAG",
)
my_first_page.add_aggregation(hist_6)

# Heatmap Engine Speed vs. torque
hist_4_name = "rpm_torque_heatmap"
hist_4_desc = "Engine RPM vs. torque heatmap within RPM events"
hist_4_x_bins = [float(i) for i in range(0, 2400, 50)]
hist_4_y_bins = [float(i) for i in range(0, 120, 5)]
hist_4 = Histogram2DDuration(
    name=hist_4_name,
    x_expr=eng_rpm,
    y_expr=veh_spd,
    event=veh_spd_event,
    x_bins=hist_4_x_bins,
    y_bins=hist_4_y_bins,
    desc=hist_4_desc,
    x_channel_name="is1_eng_speed",
    y_channel_name="can_vehicle_speed",
)
my_first_page.add_aggregation(hist_4)

histogram_7_name = "histogram_2d_custom_weights"
histogram_7_desc = "histogram 2d custom weights"
histogram_7_x_bins = [float(i) for i in range(0, 2400, 50)]
histogram_7_y_bins = [float(i) for i in range(0, 125, 5)]
histogram_7 = Histogram2DCustomWeights(
    name=histogram_7_name,
    x_expr=eng_rpm,
    y_expr=veh_spd,
    event=veh_spd_event,
    weights_expr=odo_km,
    x_bins=histogram_7_x_bins,
    y_bins=histogram_7_y_bins,
    desc=histogram_7_desc,
    x_channel_name="is1_eng_speed",
    y_channel_name="can_vehicle_speed",
    weights_channel_name="can_in_CGW_C1_TotalVehDist_Cval_DIAG",
    math_fct_for_weights="diff",
)
my_first_page.add_aggregation(histogram_7)

histogram_8_name = "histogram_2d_distance"
histogram_8_desc = "histogram 2d distance"
histogram_8_x_bins = [float(i) for i in range(0, 2400, 50)]
histogram_8_y_bins = [float(i) for i in range(0, 125, 5)]
histogram_8 = Histogram2DDistance(
    name=histogram_8_name,
    x_expr=eng_rpm,
    y_expr=veh_spd,
    event=veh_spd_event,
    weights_expr=odo_km,
    x_bins=histogram_8_x_bins,
    y_bins=histogram_8_y_bins,
    desc=histogram_8_desc,
    x_channel_name="is1_eng_speed",
    y_channel_name="can_vehicle_speed",
)
my_first_page.add_aggregation(histogram_8)

### Run Calculation and Exract Results from Gold Layer

In [0]:
my_report.determine_report()
my_report.persist_results()

hist_df = spark.read.table(f"{catalog_out}.{schema_out}.{prefix_out}_histogram_fact")
hist2d_df = spark.read.table(
    f"{catalog_out}.{schema_out}.{prefix_out}_histogram2d_fact"
)

hist_meta_df = spark.read.table(
    f"{catalog_out}.{schema_out}.{prefix_out}_histogram_dimension"
)
hist2d_meta_df = spark.read.table(
    f"{catalog_out}.{schema_out}.{prefix_out}_histogram2d_dimension"
)

event_df = spark.read.table(
    f"{catalog_out}.{schema_out}.{prefix_out}_event_instance_fact"
)
event_meta_df = spark.read.table(
    f"{catalog_out}.{schema_out}.{prefix_out}_event_dimension"
)
measurement_dim_df = spark.read.table(
    f"{catalog_out}.{schema_out}.{prefix_out}_measurement_dimension"
)